In [2]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 01 — DATASET PREPARATION
# Marathi Mitra — Vocabulary Learning App
#
# This notebook:
# → Loads raw vocabulary from vocabulary.json
# → Explores and visualizes the dataset
# → Formats into training examples
# → Validates dataset quality
# → Saves vocabulary_dataset.json for training
# ═══════════════════════════════════════════════════════════
print("📚 Marathi Mitra — Dataset Preparation")
print("=" * 50)

📚 Marathi Mitra — Dataset Preparation


In [ ]:
import json
import os
from collections import Counter

print("✅ Libraries loaded")

✅ Libraries loaded


In [4]:
# ── Load vocabulary.json ─────────────────────────────────
with open("../data/vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

print(f"✅ Loaded {len(vocabulary)} words")
print(f"\nFirst entry:")
print(json.dumps(vocabulary[0], ensure_ascii=False, indent=2))

✅ Loaded 30 words

First entry:
{
  "word": "butterfly",
  "marathi": "फुलपाखरू",
  "pronunciation": "Phul-pakh-roo",
  "sentence": "फुलपाखरू फुलांवर बसते.",
  "translation": "The butterfly sits on flowers.",
  "fun_fact": "फूल म्हणजे flower + पाखरू म्हणजे bird — flower-bird! 🦋"
}


In [5]:
# ── What fields does each entry have? ───────────────────
print("DATASET STRUCTURE")
print("=" * 50)

sample = vocabulary[0]
for key, value in sample.items():
    print(f"\n  {key}:")
    print(f"    {value}")

print(f"\n{'='*50}")
print(f"Total fields per entry: {len(sample)}")

DATASET STRUCTURE

  word:
    butterfly

  marathi:
    फुलपाखरू

  pronunciation:
    Phul-pakh-roo

  sentence:
    फुलपाखरू फुलांवर बसते.

  translation:
    The butterfly sits on flowers.

  fun_fact:
    फूल म्हणजे flower + पाखरू म्हणजे bird — flower-bird! 🦋

Total fields per entry: 6


In [8]:
# ── Analyse Devanagari script usage ──────────────────────
print("MARATHI WORD ANALYSIS")
print("=" * 50)

for item in vocabulary:
    word       = item["word"]
    marathi    = item["marathi"]
    pronoun    = item["pronunciation"]
    char_count = len(marathi)

    print(f"  {word:<15} → {marathi:<15} ({pronoun})")

print(f"\n{'─'*50}")

# Check all have Devanagari
no_devanagari = [
    item["word"] for item in vocabulary
    if not any("\u0900" <= c <= "\u097F" for c in item["marathi"])
]
if no_devanagari:
    print(f"⚠️  Missing Devanagari: {no_devanagari}")
else:
    print(f"✅ All {len(vocabulary)} words have Devanagari script")

MARATHI WORD ANALYSIS
  butterfly       → फुलपाखरू        (Phul-pakh-roo)
  sun             → सूर्य           (Soor-ya)
  moon            → चंद्र           (Chan-dra)
  rain            → पाऊस            (Paa-oos)
  flower          → फूल             (Phool)
  tree            → झाड             (zaad)
  river           → नदी             (Na-dee)
  mountain        → पर्वत           (Par-vat)
  sky             → आकाश            (Aa-kaash)
  water           → पाणी            (Paa-ni)
  cat             → मांजर           (Maan-jar)
  dog             → कुत्रा          (Koo-tra)
  bird            → पक्षी           (Pak-shi)
  fish            → मासा            (Maa-sa)
  elephant        → हत्ती           (Hat-tee)
  cow             → गाय             (Gaay)
  monkey          → माकड            (Maa-kad)
  parrot          → पोपट            (Po-pat)
  mother          → आई              (Aa-ee)
  father          → बाबा            (Baa-baa)
  sister          → बहीण            (Ba-heen)
  brother        

In [9]:
# ── Check every entry has all required fields ────────────
print("DATASET QUALITY CHECK")
print("=" * 50)

required_fields = [
    "word", "marathi", "pronunciation",
    "sentence", "translation", "fun_fact"
]

issues = []

for i, item in enumerate(vocabulary):
    for field in required_fields:
        if field not in item:
            issues.append(f"Entry {i} ({item.get('word','?')}) missing: {field}")
        elif not item[field].strip():
            issues.append(f"Entry {i} ({item.get('word','?')}) empty: {field}")

if issues:
    print("⚠️  Issues found:")
    for issue in issues:
        print(f"  → {issue}")
else:
    print(f"✅ All {len(vocabulary)} entries have all required fields")
    print(f"✅ No empty fields found")

# Check for duplicate words
words      = [item["word"] for item in vocabulary]
duplicates = [w for w, count in Counter(words).items() if count > 1]
if duplicates:
    print(f"⚠️  Duplicate words: {duplicates}")
else:
    print(f"✅ No duplicate words found")

DATASET QUALITY CHECK
✅ All 30 entries have all required fields
✅ No empty fields found
✅ No duplicate words found


In [10]:
# ── Show what the model actually trains on ───────────────
INSTRUCTION = (
    "You are Marathi Mitra, a friendly Marathi teacher for kids. "
    "When given an English word, teach it in Marathi with the word "
    "in Devanagari script, pronunciation, a simple story sentence, "
    "and a fun fact. Always be encouraging and kid-friendly."
)

def format_output(item):
    return f"""🌟 **{item['word'].upper()}** in Marathi is **{item['marathi']}**

📢 **How to say it:** {item['pronunciation']}

📖 **Example sentence:**
{item['sentence']}
*({item['translation']})*

🎉 **Fun Fact:** {item['fun_fact']}"""


def format_training_example(item):
    input_text  = f"Teach me the Marathi word for: {item['word']}"
    output_text = format_output(item)
    return {
        "word":        item["word"],
        "marathi":     item["marathi"],
        "instruction": INSTRUCTION,
        "input":       input_text,
        "output":      output_text,
        "text": f"""### Instruction:
{INSTRUCTION}

### Input:
{input_text}

### Response:
{output_text}"""
    }


print("TRAINING FORMAT EXAMPLE")
print("=" * 50)
print("This is exactly what the model sees during training:")
print("─" * 50)
sample = format_training_example(vocabulary[0])
print(sample["text"])
print("─" * 50)
print(f"\nTotal characters: {len(sample['text'])}")
print(f"Approximate tokens: ~{len(sample['text'])//4}")

TRAINING FORMAT EXAMPLE
This is exactly what the model sees during training:
──────────────────────────────────────────────────
### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. When given an English word, teach it in Marathi with the word in Devanagari script, pronunciation, a simple story sentence, and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: butterfly

### Response:
🌟 **BUTTERFLY** in Marathi is **फुलपाखरू**

📢 **How to say it:** Phul-pakh-roo

📖 **Example sentence:**
फुलपाखरू फुलांवर बसते.
*(The butterfly sits on flowers.)*

🎉 **Fun Fact:** फूल म्हणजे flower + पाखरू म्हणजे bird — flower-bird! 🦋
──────────────────────────────────────────────────

Total characters: 558
Approximate tokens: ~139


In [11]:
# ── Check token lengths to ensure fit in max_seq_length ──
print("TOKEN LENGTH ANALYSIS")
print("=" * 50)
print("(Approximate — actual tokens depend on tokenizer)")
print()

examples = [format_training_example(item) for item in vocabulary]
lengths  = [len(ex["text"]) // 4 for ex in examples]  # rough estimate

print(f"  Shortest example: ~{min(lengths)} tokens")
print(f"  Longest example:  ~{max(lengths)} tokens")
print(f"  Average:          ~{sum(lengths)//len(lengths)} tokens")
print(f"  Our max_seq_length: 512 tokens")
print()

too_long = [ex["word"] for ex, l in zip(examples, lengths) if l > 512]
if too_long:
    print(f"⚠️  May exceed 512 tokens: {too_long}")
else:
    print(f"✅ All examples fit within 512 token limit")

TOKEN LENGTH ANALYSIS
(Approximate — actual tokens depend on tokenizer)

  Shortest example: ~122 tokens
  Longest example:  ~146 tokens
  Average:          ~128 tokens
  Our max_seq_length: 512 tokens

✅ All examples fit within 512 token limit


In [12]:
# ── Generate and save vocabulary_dataset.json ────────────
examples    = [format_training_example(item) for item in vocabulary]
output_path = "../data/vocabulary_dataset.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(examples)} training examples")
print(f"✅ Location: {output_path}")
print(f"\nFields in each training example:")
for key in examples[0].keys():
    print(f"  → {key}")

✅ Saved 30 training examples
✅ Location: ../data/vocabulary_dataset.json

Fields in each training example:
  → word
  → marathi
  → instruction
  → input
  → output
  → text
